## Step 1: Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import glob
import os


In [5]:

pm25_files = glob.glob("../../data/raw/PM25_*.csv")

print("Files found:", pm25_files)

pm25_list = []

for file in pm25_files:
    try:
        df = pd.read_csv(file)
        pm25_list.append(df)
        print("Loaded:", file, df.shape)
    except Exception as e:
        print("FAILED:", file)
        print("Error:", e)


Files found: ['../../data/raw\\PM25_2020.csv', '../../data/raw\\PM25_2021.csv', '../../data/raw\\PM25_2022.csv', '../../data/raw\\PM25_2023.csv']
FAILED: ../../data/raw\PM25_2020.csv
Error: Error tokenizing data. C error: Expected 2 fields in line 8, saw 32

FAILED: ../../data/raw\PM25_2021.csv
Error: Error tokenizing data. C error: Expected 2 fields in line 8, saw 32

FAILED: ../../data/raw\PM25_2022.csv
Error: Error tokenizing data. C error: Expected 2 fields in line 8, saw 32

Loaded: ../../data/raw\PM25_2023.csv (109872, 32)


C:\Users\hetac\AppData\Local\Temp\ipykernel_47380\24654262.py:9: DtypeWarning: Columns (1,2,5,6,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


## Step 2: Load Raw PM2.5 Files (Skip Metadata Rows)

In [7]:
pm25_files = glob.glob("../../data/raw/PM25_*.csv")

pm25_list = []
for f in pm25_files:
    df = pd.read_csv(f, skiprows=7, encoding="latin1")
    pm25_list.append(df)
    print("Loaded:", f, df.shape)

pm25_raw = pd.concat(pm25_list, ignore_index=True)
print("PM25 combined shape:", pm25_raw.shape)
pm25_raw.head()


Loaded: ../../data/raw\PM25_2020.csv (92964, 32)
Loaded: ../../data/raw\PM25_2021.csv (85410, 32)
Loaded: ../../data/raw\PM25_2022.csv (98185, 32)
Loaded: ../../data/raw\PM25_2023.csv (109865, 32)
PM25 combined shape: (386424, 32)


,Pollutant//Polluant,Method Code//Code MÃ©thode,NAPS ID//Identifiant SNPA,City//Ville,Province/Territory//Province/Territoire,Latitude//Latitude,Longitude//Longitude,Date//Date,H01//H01,H02//H02,...,H15//H15,H16//H16,H17//H17,H18//H18,H19//H19,H20//H20,H21//H21,H22//H22,H23//H23,H24//H24
0,PM2.5,184,50103,MontrÃ©al,QC,45.64103,-73.49968,2020-01-01,9,6,...,3,4,5,6,6,5,4,2,2,2
1,PM2.5,184,50103,MontrÃ©al,QC,45.64103,-73.49968,2020-01-02,2,3,...,10,9,8,8,8,8,8,9,8,7
2,PM2.5,184,50103,MontrÃ©al,QC,45.64103,-73.49968,2020-01-03,8,8,...,15,14,17,17,16,17,14,14,11,7
3,PM2.5,184,50103,MontrÃ©al,QC,45.64103,-73.49968,2020-01-04,7,5,...,4,5,3,4,5,5,5,4,4,4
4,PM2.5,184,50103,MontrÃ©al,QC,45.64103,-73.49968,2020-01-05,4,5,...,2,4,4,4,5,5,6,9,9,10


## Step 3: Clean Column Names

In [9]:
pm25_raw.columns = [c.split("/")[0].strip() for c in pm25_raw.columns]
print(pm25_raw.columns)

Index(['Pollutant', 'Method Code', 'NAPS ID', 'City', 'Province', 'Latitude',
       'Longitude', 'Date', 'H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07',
       'H08', 'H09', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16', 'H17',
       'H18', 'H19', 'H20', 'H21', 'H22', 'H23', 'H24'],
      dtype='object')


## Step 4: Replace -999 With NaN in Hourly Columns

In [11]:
hour_cols = [c for c in pm25_raw.columns if c.startswith("H")]

pm25_raw[hour_cols] = pm25_raw[hour_cols].replace(-999, np.nan)
pm25_raw["PM25_daily"] = pm25_raw[hour_cols].mean(axis=1)

## Step 5: Compute Daily PM2.5 Average From Hourly Values

In [20]:
pm25_raw["PM25_daily"] = pm25_raw[hour_cols].mean(axis=1)


## Step 6: Convert Date Column to Datetime

In [23]:
pm25_raw["Date"] = pd.to_datetime(pm25_raw["Date"], errors="coerce")


## Step 7: Create Clean City-Day Dataset

In [26]:
pm25_city = pm25_raw[[
    "City",
    "Date",
    "PM25_daily"
]].copy()

pm25_city["Year"] = pm25_city["Date"].dt.year
pm25_city["Month"] = pm25_city["Date"].dt.month

pm25_city.head()

,City,Date,PM25_daily,Year,Month
0,MontrÃ©al,2020-01-01,4.333333,2020,1
1,MontrÃ©al,2020-01-02,7.625000,2020,1
2,MontrÃ©al,2020-01-03,10.750000,2020,1
3,MontrÃ©al,2020-01-04,4.416667,2020,1
4,MontrÃ©al,2020-01-05,3.833333,2020,1


## Step 8: Remove Missing Daily Values

In [29]:
pm25_city = pm25_city.dropna(subset=["PM25_daily"])


## Step 9: Remove Duplicates (City-Date Level)

In [32]:
pm25_city = pm25_city.drop_duplicates(subset=["City", "Date"])

print("Duplicates:", pm25_city.duplicated(subset=["City","Date"]).sum())


Duplicates: 0


## Step 10: Save Validated Daily Dataset

In [35]:
os.makedirs("../../data/validated", exist_ok=True)

pm25_city.to_csv("../../data/validated/PM25_cityday.csv", index=False)

print("Saved: PM25_cityday.csv")

Saved: PM25_cityday.csv
